# Differential correlation — DGCA

Classical differential analysis (`metabol.differential`) asks *which
metabolites change in abundance?* Differential **correlation** asks a
complementary question: *which metabolite–metabolite relationships
change between conditions?* Two metabolites might have identical mean
levels in both groups yet be tightly co-regulated in one and
uncorrelated in the other — evidence for a rewired pathway.

**`ov.metabol.dgca`** implements DGCA (McKenzie et al., *BMC
Genomics* 2016): Fisher-z transformation of each pair's correlation
in each condition, z-test on the difference, BH FDR, and a
categorical *DC class* label (`+/+`, `+/0`, `+/-`, …).

We use the **Cachexia** dataset — 77 urinary metabolomes labelled
`cachexic` vs `control` (47 vs 30).


## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import omicverse as ov

csv_path = ov.datasets.download_data(
    url='https://rest.xialab.ca/api/download/metaboanalyst/human_cachexia.csv',
    file_path='human_cachexia.csv',
    dir='metabol_demo',
)
adata = ov.metabol.read_metaboanalyst(csv_path, group_col='Muscle loss')
adata = ov.metabol.impute(adata, method='qrilc', seed=0)
adata = ov.metabol.normalize(adata, method='pqn')
adata = ov.metabol.transform(adata, method='log')
adata


## 1 — Run DGCA

For 63 metabolites we get `63*62/2 = 1953` pairs. Use Spearman for
metabolomics (heavy-tailed distributions) and threshold |r|≥0.3 for
class labels (the default, matching MetaboAnalyst's network module).


In [ ]:
dc = ov.metabol.dgca(
    adata,
    group_col='Muscle loss',
    group_a='cachexic', group_b='control',
    method='spearman',
    abs_r_threshold=0.3,
)
print(f'{len(dc)} pairs tested')
dc.head(15)


## 2 — DC class distribution

Each pair gets a two-symbol label `A/B` where `+/-/0` summarise the
correlation strength in groups cachexic (`A`) and control (`B`):


In [ ]:
class_counts = dc['dc_class'].value_counts()
class_counts


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
class_counts.plot.bar(ax=ax, color='C0', edgecolor='k')
ax.set_ylabel('Number of pairs')
ax.set_xlabel('DC class (A=cachexic / B=control)')
ax.set_yscale('log')
plt.xticks(rotation=0)
fig.tight_layout()
plt.show()


## 3 — Significantly rewired pairs

Filter by `padj < 0.05`, sort by |z_diff|, inspect the top
differentially correlated pairs:


In [ ]:
sig = dc[dc['padj'] < 0.05].copy()
sig['abs_z'] = sig['z_diff'].abs()
top = sig.sort_values('abs_z', ascending=False).head(15)
top[['feature_a', 'feature_b', 'r_a', 'r_b', 'z_diff', 'padj', 'dc_class']]


## 4 — Visualise one rewired pair

Pick the top pair and show the raw correlation in each group on a
scatter plot:


In [ ]:
top_row = top.iloc[0]
fa, fb = top_row['feature_a'], top_row['feature_b']
idx_a = list(adata.var_names).index(fa)
idx_b = list(adata.var_names).index(fb)

groups = adata.obs['Muscle loss'].astype(str).to_numpy()
X = np.asarray(adata.X)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, label, colour in zip(axes, ['cachexic', 'control'], ['C3', 'C0']):
    mask = groups == label
    ax.scatter(X[mask, idx_a], X[mask, idx_b], color=colour,
               edgecolor='k', s=40)
    ax.set_xlabel(fa)
    ax.set_title(f'{label}  (r = {(top_row["r_a"] if label == "cachexic" else top_row["r_b"]):.2f})')
axes[0].set_ylabel(fb)
fig.tight_layout()
plt.show()


## 5 — Rewired correlation network

Build a small NetworkX graph of the top-rewired pairs coloured by DC
class:


In [ ]:
import networkx as nx

top50 = sig.sort_values('abs_z', ascending=False).head(50)
G = nx.Graph()
class_colour = {'+/+': 'C2', '-/-': 'C4', '+/-': 'C3', '-/+': 'C3',
                '+/0': 'C0', '0/+': 'C1', '-/0': 'C5', '0/-': 'C6', '0/0': 'lightgray'}
for _, r in top50.iterrows():
    G.add_edge(r['feature_a'], r['feature_b'],
               weight=abs(r['z_diff']),
               color=class_colour.get(r['dc_class'], 'gray'))
pos = nx.spring_layout(G, seed=0, k=0.7)
edge_colors = [G[u][v]['color'] for u, v in G.edges()]
fig, ax = plt.subplots(figsize=(8, 6))
nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=1.5, alpha=0.8)
nx.draw_networkx_nodes(G, pos, node_size=280, node_color='white',
                        edgecolors='k', ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
ax.set_title('Top 50 rewired pairs — edge colour = DC class')
ax.axis('off')
fig.tight_layout()
plt.show()


## Takeaways

- DGCA surfaces relationships that `metabol.differential` misses.
- The **class labels** (`+/+`, `+/0`, `+/-`…) give an interpretable
  summary: `+/0` = "correlation gained in A", `+/-` = "inversion".
- For large panels (`p > 500`) restrict with `features=<top-N by
  univariate AUC or VIP>` to keep the O(p²) output manageable.

DGCA pairs naturally with ASCA + biomarker panels: they answer
different questions about the same data.
